[Reference](https://pub.towardsai.net/how-to-combine-graph-relationships-and-ai-embeddings-in-neo4j-for-smarter-search-17f7aa6a2408)

# Building a Movie Knowledge Graph: Setting Up Neo4j and Loading Data
Neo4j Setup: [link](https://neo4j.com/docs/)

In [1]:
!pip install langchain-community neo4j python-dotenv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 325.4/325.4 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 31.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 471.5/471.5 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 1.6 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 0.3.79
    Uninstalling langchain-core-0.3.79:
      Successfully uninstalled langchain-core-0.3.79
  Attempting uninstall: langchain-text-splitters
    Found existing installation: langchain-text-splitters 0.3.11
    Uninstalling langchain-text-splitters-0.3.11:
      Successf

In [2]:
from langchain_community.graphs import Neo4jGraph
import os

NEO4J_URI = os.getenv('NEO4J_URI')
NEO4J_USERNAME = os.getenv('NEO4J_USERNAME')
NEO4J_PASSWORD = os.getenv('NEO4J_PASSWORD')
NEO4J_DATABASE = os.getenv('NEO4J_DATABASE') or 'neo4j'

graph = Neo4jGraph(
    url=NEO4J_URI,
    username=NEO4J_USERNAME,
    password=NEO4J_PASSWORD,
    database=NEO4J_DATABASE
)

In [3]:
create_query = """
CREATE
  (nolan:Person {name:"Christopher Nolan"}),
  (leo:Person {name:"Leonardo DiCaprio"}),
  (matt:Person {name:"Matthew McConaughey"}),
  (anne:Person {name:"Anne Hathaway"}),

  (inception:Movie {id:1, title:"Inception", description:"A mind-bending thriller about dream infiltration.", year:2010}),
  (interstellar:Movie {id:2, title:"Interstellar", description:"A space epic exploring black holes and time dilation.", year:2014}),
  (darkknight:Movie {id:3, title:"The Dark Knight", description:"Batman faces chaos as the Joker terrorizes Gotham.", year:2008}),

  (action:Genre {name:"Action"}),
  (sci_fi:Genre {name:"Sci-Fi"}),

  (nolan)-[:DIRECTED]->(inception),
  (nolan)-[:DIRECTED]->(interstellar),
  (nolan)-[:DIRECTED]->(darkknight),

  (leo)-[:ACTED_IN]->(inception),
  (matt)-[:ACTED_IN]->(interstellar),
  (anne)-[:ACTED_IN]->(interstellar),

  (inception)-[:IN_GENRE]->(sci_fi),
  (interstellar)-[:IN_GENRE]->(sci_fi),
  (darkknight)-[:IN_GENRE]->(action);
"""


graph.query(create_query)
print("Movie graph created!")

In [4]:
query = """
MATCH (p:Person)-[:DIRECTED]->(m:Movie)-[:IN_GENRE]->(g:Genre {name:"Sci-Fi"})
RETURN DISTINCT p.name AS Director;
"""
result = graph.query(query)
print(result)

# Using Embeddings in Neo4j for Smarter Retrieval

In [5]:
!pip install langchain-openai

from openai import OpenAI
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Neo4jVector
from langchain_community.graphs import Neo4jGraph

vector_index = Neo4jVector.from_existing_graph(
    embedding=OpenAIEmbeddings(api_key=api_key, model="text-embedding-3-small"),
    url=NEO4J_URI,
    username=NEO4J_USERNAME,
    password=NEO4J_PASSWORD,
    index_name="movie_index",
    node_label="Movie",
    text_node_properties=["title", "description"],
    embedding_node_property="embedding",
)

print("Embeddings created and stored in Neo4j!")

In [6]:
query = "movies about dreams or subconscious mind"
results = vector_index.similarity_search(query, k=3)

for r in results:
    print(r.page_content)

# Hybrid Retrieval-Combining Semantic + Graph Search :

In [7]:
def hybrid_search(question, k=3):
    results = vector_index.similarity_search(question, k=k)

    titles = [
        line.replace("title:", "").strip()
        for r in results
        for line in r.page_content.split("\n")
        if line.lower().startswith("title:")
    ]

    if not titles:
        print("No titles found from vector search.")
        return []

    print("Similar movies found:", titles)

    cypher = f"""
    MATCH (m:Movie)-[:IN_GENRE]->(g:Genre),
          (p:Person)-[:DIRECTED]->(m)
    WHERE m.title IN {titles}
    RETURN m.title AS Movie,
           collect(DISTINCT p.name) AS Directors,
           collect(DISTINCT g.name) AS Genres
    ORDER BY m.title
    """

    result = graph.query(cypher)
    return result

hybrid_search("movies about dreams or subconscious mind")